In [0]:
base_path = "/Volumes/workspace/default/my_volume/flight_project"
silver_path      = f"{base_path}/silver/flights"
gold_airline_path = f"{base_path}/gold/airline_summary"
gold_cascade_path = f"{base_path}/gold/cascade_impact"

df_silver = spark.read.format("delta").load(silver_path)
print(f"✅ Silver loaded: {df_silver.count()} rows")

✅ Silver loaded: 12073 rows


In [0]:
from pyspark.sql.functions import *

df_gold_airline = df_silver.groupBy("airline_code").agg(
    count("icao24").alias("total_flights"),
    sum(when(col("is_delayed") == True, 1).otherwise(0)).alias("delayed_flights"),
    round(avg("delay_minutes"), 2).alias("avg_delay_minutes"),
    round(avg("delay_score"), 2).alias("avg_delay_score"),
    round(
        sum(when(col("is_delayed") == True, 1).otherwise(0)) * 100.0 / count("icao24"),
        2
    ).alias("delay_percentage")
).orderBy("delayed_flights", ascending=False)

df_gold_airline.show(10)

+------------+-------------+---------------+-----------------+---------------+----------------+
|airline_code|total_flights|delayed_flights|avg_delay_minutes|avg_delay_score|delay_percentage|
+------------+-------------+---------------+-----------------+---------------+----------------+
|         CXK|           63|             32|            22.86|          40.32|           50.79|
|         UAL|          453|             23|             1.02|           2.74|            5.08|
|         N21|           70|             22|            14.14|          30.57|           31.43|
|         N73|           75|             22|             13.2|          30.67|           29.33|
|            |          101|             21|             7.87|           19.6|           20.79|
|         SWA|          406|             20|             0.99|            3.2|            4.93|
|         SKW|          189|             17|             2.33|           5.61|            8.99|
|         N12|           60|            

In [0]:
df_gold_airline.write.format("delta") \
    .mode("overwrite") \
    .save(gold_airline_path)

print("✅ Gold Airline Summary saved!")

✅ Gold Airline Summary saved!


In [0]:
# Same nearest_airport close flights = potential cascade
from pyspark.sql.window import Window

# Delayed flights
df_delayed = df_silver.filter(col("is_delayed") == True) \
    .select("icao24", "callsign", "nearest_airport", 
            "delay_minutes", "airline_code")

# Same airport remaining flights —  affect
df_affected = df_silver.filter(col("is_delayed") == False) \
    .select(
        col("icao24").alias("affected_icao"),
        col("callsign").alias("affected_callsign"),
        col("nearest_airport").alias("affected_airport"),
        col("airline_code").alias("affected_airline")
    )

# JOIN — same airport pe
df_cascade = df_delayed.join(
    df_affected,
    df_delayed["nearest_airport"] == df_affected["affected_airport"],
    "inner"
).groupBy(
    "icao24", "callsign", "nearest_airport", "delay_minutes"
).agg(
    count("affected_icao").alias("flights_potentially_affected")
).orderBy("flights_potentially_affected", ascending=False)

df_cascade.show(10)

+------+--------+---------------+-------------+----------------------------+
|icao24|callsign|nearest_airport|delay_minutes|flights_potentially_affected|
+------+--------+---------------+-------------+----------------------------+
|a2795f|  EPI259|            DAB|           45|                          53|
|a8214d|  BPX207|            DAB|           45|                          53|
|a8fe29|  LFA536|            DAB|           45|                          53|
|a2e575|  N286ND|            AUS|           45|                          29|
|a6a64a|  N5277P|            AUS|           45|                          29|
|ab9154|  N8442A|            AUS|           45|                          29|
|abe82c|  SWA988|            SFB|           20|                          27|
|a944ea|  N697CA|            RBD|           45|                          27|
|a919b4|  N686CA|            RBD|           45|                          27|
|a1d825|  N218KK|            RBD|           45|                          27|

In [0]:
df_cascade.write.format("delta") \
    .mode("overwrite") \
    .save(gold_cascade_path)

print("✅ Gold Cascade Impact saved!")

✅ Gold Cascade Impact saved!


In [0]:
df_g_airline = spark.read.format("delta").load(gold_airline_path)
df_g_cascade = spark.read.format("delta").load(gold_cascade_path)

print("=== GOLD SUMMARY ===\n")
print(f"✅ Gold Airline Summary: {df_g_airline.count()} airlines")
print(f"✅ Gold Cascade Impact:  {df_g_cascade.count()} delayed flights\n")

print("--- Top 5 Most Delayed Airlines ---")
df_g_airline.orderBy("delay_percentage", ascending=False).show(5)

print("--- Top 5 Cascade Impact ---")
df_g_cascade.orderBy("flights_potentially_affected", ascending=False).show(5)

=== GOLD SUMMARY ===

✅ Gold Airline Summary: 1389 airlines
✅ Gold Cascade Impact:  1458 delayed flights

--- Top 5 Most Delayed Airlines ---
+------------+-------------+---------------+-----------------+---------------+----------------+
|airline_code|total_flights|delayed_flights|avg_delay_minutes|avg_delay_score|delay_percentage|
+------------+-------------+---------------+-----------------+---------------+----------------+
|         HBA|            3|              3|             45.0|           50.0|           100.0|
|         DMM|            2|              2|             45.0|           60.0|           100.0|
|         LOO|            2|              2|             32.5|           50.0|           100.0|
|         PCM|            2|              2|             45.0|           60.0|           100.0|
|         MVK|            2|              2|             45.0|           60.0|           100.0|
+------------+-------------+---------------+-----------------+---------------+------------

In [0]:
df_gold_airline = df_silver.groupBy("airline_code").agg(
    count("icao24").alias("total_flights"),
    sum(when(col("is_delayed") == True, 1).otherwise(0)).alias("delayed_flights"),
    round(avg("delay_minutes"), 2).alias("avg_delay_minutes"),
    round(avg("delay_score"), 2).alias("avg_delay_score"),
    round(
        sum(when(col("is_delayed") == True, 1).otherwise(0)) * 100.0 / count("icao24"),
        2
    ).alias("delay_percentage")
).filter(
    col("total_flights") >= 10  # minimum 10 flights wali airlines
).orderBy("delayed_flights", ascending=False)

df_gold_airline.show(10)

+------------+-------------+---------------+-----------------+---------------+----------------+
|airline_code|total_flights|delayed_flights|avg_delay_minutes|avg_delay_score|delay_percentage|
+------------+-------------+---------------+-----------------+---------------+----------------+
|         CXK|           63|             32|            22.86|          40.32|           50.79|
|         UAL|          453|             23|             1.02|           2.74|            5.08|
|         N21|           70|             22|            14.14|          30.57|           31.43|
|         N73|           75|             22|             13.2|          30.67|           29.33|
|            |          101|             21|             7.87|           19.6|           20.79|
|         SWA|          406|             20|             0.99|            3.2|            4.93|
|         SKW|          189|             17|             2.33|           5.61|            8.99|
|         N12|           60|            

In [0]:
df_gold_airline = df_silver.groupBy("airline_code").agg(
    count("icao24").alias("total_flights"),
    sum(when(col("is_delayed") == True, 1).otherwise(0)).alias("delayed_flights"),
    round(avg("delay_minutes"), 2).alias("avg_delay_minutes"),
    round(avg("delay_score"), 2).alias("avg_delay_score"),
    round(
        sum(when(col("is_delayed") == True, 1).otherwise(0)) * 100.0 / count("icao24"),
        2
    ).alias("delay_percentage")
).filter(
    (col("total_flights") >= 10) &
    (col("airline_code").isNotNull()) &
    (trim(col("airline_code")) != "")
).orderBy("delayed_flights", ascending=False)

df_gold_airline.show(10)

+------------+-------------+---------------+-----------------+---------------+----------------+
|airline_code|total_flights|delayed_flights|avg_delay_minutes|avg_delay_score|delay_percentage|
+------------+-------------+---------------+-----------------+---------------+----------------+
|         CXK|           63|             32|            22.86|          40.32|           50.79|
|         UAL|          453|             23|             1.02|           2.74|            5.08|
|         N73|           75|             22|             13.2|          30.67|           29.33|
|         N21|           70|             22|            14.14|          30.57|           31.43|
|         SWA|          406|             20|             0.99|            3.2|            4.93|
|         N12|           60|             17|            12.75|          27.17|           28.33|
|         SKW|          189|             17|             2.33|           5.61|            8.99|
|         AAL|          437|            

In [0]:
df_gold_airline.write.format("delta") \
    .mode("overwrite") \
    .save(gold_airline_path)

print("✅ Gold Airline Summary saved!")

✅ Gold Airline Summary saved!
